In [17]:
#--- 라이브러리 불러오기 + DB 연결 + CSV 적재 + 기본 확인
import pandas as pd
import sqlite3
import os

# 현재 작업 경로 확인
print("현재 작업 폴더:", os.getcwd())

# DB 연결
conn = sqlite3.connect("../db/olist.db")

# orders CSV 불러오기
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

# orders 테이블 저장
orders.to_sql("orders", conn, if_exists="replace", index=False)

# 테이블 목록 확인
print(pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn))

# orders 데이터 미리보기
pd.read_sql("""
SELECT *
FROM orders
LIMIT 5;
""", conn)

현재 작업 폴더: /Users/chaeyoung/Documents/GitHub/olist-ecommerce-analysis/notebooks
                                name
0                          customers
1                        geolocation
2                        order_items
3                     order_payments
4                      order_reviews
5                             orders
6  product_category_name_translation
7                           products
8                            sellers
9                         test_table


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [18]:
#--- 주문 기본 현황 분석

# 전체 주문 수
display(pd.read_sql("""
SELECT COUNT(*) AS total_orders
FROM orders;
""", conn))

# 주문 상태 분포
display(pd.read_sql("""
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
""", conn))

# 주문 상태 비율
display(pd.read_sql("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders),
        2
    ) AS order_pct
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
""", conn))

# 월별 주문 수 추이
display(pd.read_sql("""
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_month
ORDER BY order_month;
""", conn))

# 일별 주문 수 추이
display(pd.read_sql("""
SELECT
    date(order_purchase_timestamp) AS order_date,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_date
ORDER BY order_date;
""", conn))

,total_orders
0,99441


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


,order_status,order_count,order_pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


,order_month,order_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


,order_date,order_count
0,2016-09-04,1
1,2016-09-05,1
2,2016-09-13,1
3,2016-09-15,1
4,2016-10-02,1
...,...,...
629,2018-09-29,1
630,2018-10-01,1
631,2018-10-03,1
632,2018-10-16,1


In [19]:
#-- 배송 시간 분석

# 주문 승인까지 평균 소요 일수
display(pd.read_sql("""
SELECT
    ROUND(
        AVG(julianday(order_approved_at) - julianday(order_purchase_timestamp)),
        2
    ) AS avg_approval_days
FROM orders
WHERE order_approved_at IS NOT NULL;
""", conn))

# 구매 후 배송 시작까지 평균 소요 일수
display(pd.read_sql("""
SELECT
    ROUND(
        AVG(julianday(order_delivered_carrier_date) - julianday(order_purchase_timestamp)),
        2
    ) AS avg_to_carrier_days
FROM orders
WHERE order_delivered_carrier_date IS NOT NULL;
""", conn))

# 구매 후 고객 배송 완료까지 평균 소요 일수
display(pd.read_sql("""
SELECT
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)),
        2
    ) AS avg_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
""", conn))

# 배송 시작 후 배송 완료까지 평균 소요 일수
display(pd.read_sql("""
SELECT
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_delivered_carrier_date)),
        2
    ) AS avg_carrier_to_customer_days
FROM orders
WHERE order_delivered_carrier_date IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL;
""", conn))

# 배송 지연 주문 수
display(pd.read_sql("""
SELECT
    COUNT(*) AS delayed_orders
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND order_delivered_customer_date > order_estimated_delivery_date;
""", conn))

# 배송 지연 비율
display(pd.read_sql("""
SELECT
    COUNT(*) AS delayed_orders,
    ROUND(
        COUNT(*) * 100.0 / (
            SELECT COUNT(*)
            FROM orders
            WHERE order_delivered_customer_date IS NOT NULL
              AND order_estimated_delivery_date IS NOT NULL
        ),
        2
    ) AS delayed_order_pct
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND order_delivered_customer_date > order_estimated_delivery_date;
""", conn))

# 평균 지연 일수
display(pd.read_sql("""
SELECT
    ROUND(
        AVG(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date)),
        2
    ) AS avg_delay_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND order_delivered_customer_date > order_estimated_delivery_date;
""", conn))

,avg_approval_days
0,0.43


,avg_to_carrier_days
0,3.23


,avg_delivery_days
0,12.56


,avg_carrier_to_customer_days
0,9.33


,delayed_orders
0,7827


,delayed_orders,delayed_order_pct
0,7827,8.11


,avg_delay_days
0,9.55


In [21]:
#--결측치 및 상태별 확인

# 주요 날짜 컬럼 결측치 확인
display(pd.read_sql("""
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_order_approved_at,
    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_delivered_carrier_date,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_delivered_customer_date,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS null_estimated_delivery_date
FROM orders;
""", conn))

# 주문 상태별 결측치 확인
display(pd.read_sql("""
SELECT
    order_status,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_order_approved_at,
    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_delivered_carrier_date,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_delivered_customer_date
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
""", conn))

# 배송 완료 주문 기준 최소/평균/최대 배송일
display(pd.read_sql("""
SELECT
    ROUND(MIN(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)), 2) AS min_delivery_days,
    ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)), 2) AS avg_delivery_days,
    ROUND(MAX(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)), 2) AS max_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
""", conn))

,total_orders,null_order_approved_at,null_delivered_carrier_date,null_delivered_customer_date,null_estimated_delivery_date
0,99441,160,1783,2965,0


,order_status,total_orders,null_order_approved_at,null_delivered_carrier_date,null_delivered_customer_date
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


,min_delivery_days,avg_delivery_days,max_delivery_days
0,0.53,12.56,209.63
